In [ ]:
# Import necessary libraries


import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import MultinomialNB
import os, types
from botocore.client import Config
import ibm_boto3
from ibm_watson_machine_learning import APIClient
import ast
import json

In [ ]:

import os, types
import pandas as pd
from botocore.client import Config
import ibm_boto3

def __iter__(self): return 0

# @hidden_cell
# The following code accesses a file in your IBM Cloud Object Storage. It includes your credentials.
# You might want to remove those credentials before you share the notebook.
cos_client = ibm_boto3.client(service_name='s3',
    ibm_api_key_id='HfjmkDQ5vWAr_zPsfU9uTM7wBE5LfIzwTjG5abrXDsaT',
    ibm_auth_endpoint="https://iam.cloud.ibm.com/oidc/token",
    config=Config(signature_version='oauth'),
    endpoint_url='https://s3.private.eu-de.cloud-object-storage.appdomain.cloud')

bucket = 'mediamonitoring-donotdelete-pr-il1ydrrkdjxjxm'
object_key = 'BBC News Train.csv'

body = cos_client.get_object(Bucket=bucket,Key=object_key)['Body']
# add missing __iter__ method, so pandas accepts body as file-like object
if not hasattr(body, "__iter__"): body.__iter__ = types.MethodType( __iter__, body )

data = pd.read_csv(body)
data.head()



In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data['Text'], data['Category'], test_size=0.2, random_state=42)

# Text preprocessing using TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)



In [ ]:
# Initialize and train the SVC classifier
svc_classifier = SVC(kernel='linear', C=1.0)
svc_classifier.fit(X_train_tfidf, y_train)

# Predict labels on the test set
y_pred = svc_classifier.predict(X_test_tfidf)

# Evaluate the classifier
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Initialize and train the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_tfidf, y_train)

# Predict labels on the test set
y_pred = rf_classifier.predict(X_test_tfidf)

# Evaluate the classifier
print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Initialize and train the Multinomial Naive Bayes Classifier
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train_tfidf, y_train)

# Predict labels on the test set for Multinomial Naive Bayes Classifier
y_pred_nb = nb_classifier.predict(X_test_tfidf)


# Evaluate the Multinomial Naive Bayes Classifier
print("Multinomial Naive Bayes Classifier Classification Report:")
print(classification_report(y_test, y_pred_nb))

In [ ]:
# Save the trained model to a file (e.g., using joblib or pickle)
import joblib
joblib.dump(svc_classifier, 'svc_model.pkl')

In [ ]:
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
import os, types
from botocore.client import Config
import ibm_boto3
from ibm_watson_machine_learning import APIClient
import ast
import json


In [ ]:

df=data
data.columns

In [ ]:
# preprocessing

# df["Category"] = df["Category"].apply(lambda x: ast.literal_eval(x))
y = df["Category"]

X = df["Text"]

In [ ]:
# models

mlb = MultiLabelBinarizer()
multilabel_y = mlb.fit_transform(y)
svc = LinearSVC()
clf = OneVsRestClassifier(svc)

In [ ]:
pipeline = Pipeline([('vect', CountVectorizer(ngram_range=(1, 3))), 
                    ('tfidf', TfidfTransformer()), 
                    ('clf', OneVsRestClassifier(clf))])

In [ ]:
# Use multilabel classes to fit the pipeline.
pipeline.fit(X, multilabel_y)

In [ ]:
# If X is a Pandas Series
# X = X.to_numpy()

In [ ]:
type(X)

In [ ]:
wml_credentials = {
    "apikey": '<api_key>',
    "url": 'https://eu-de.ml.cloud.ibm.com',}


In [ ]:
client = APIClient(wml_credentials)

In [ ]:

client.spaces.list()

In [ ]:
space_id = 'e924028a-bde6-4f8c-acf2-cdb0db851228'

In [ ]:
client.set.default_space(space_id)

In [ ]:
MODEL_NAME = 'MultiLabelModel'
DEPLOYMNET_NAME = 'LinearSVC_Deploy'
BEST_MODEL = clf

In [ ]:
sofware_spec_uid = client.software_specifications.get_id_by_name("runtime-22.2-py3.10")


In [ ]:
metadata = {client.repository.ModelMetaNames.NAME: 'MultiLabelModel',
            client.repository.ModelMetaNames.TYPE: 'scikit-learn_1.1',
            client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: sofware_spec_uid}

published_model = client.repository.store_model(
    model=pipeline,
    meta_props=metadata,
    training_data=X,
    training_target=multilabel_y)

In [ ]:
type(multilabel_y)

In [ ]:
published_model_uid = client.repository.get_model_id(published_model)
model_details = client.repository.get_details(published_model_uid)
print(json.dumps(model_details, indent=2))

In [ ]:
models_details = client.repository.list_models()

In [ ]:
metadata = {client.deployments.ConfigurationMetaNames.NAME: "Deployment of scikit model",
            client.deployments.ConfigurationMetaNames.ONLINE: {}}

created_deployment = client.deployments.create(published_model_uid, meta_props=metadata)